# Propose an Initial Config

`autoencodix.propose_initial_config()` gives you a strong starting hyperparameter config for
`Vanillix`, `Varix`, `Ontix`, and `Disentanglix` instead of hand-picking one or accepting the
generic schema defaults.

Under the hood, it's a lookup into a small ranked portfolio computed **offline** via Syne Tune's
zero-shot transfer-learning tooling against the BBOmix benchmark (105,000 autoencodix training
runs across 5 architectures, 2 datasets, and many downstream tasks). No optimizer runs at call
time, and `syne-tune` is **not** a runtime dependency of this package. This function is a plain
JSON lookup + Pydantic validation.

See `benchmarking/tuning/generate_initial_configs.py` for how the shipped portfolio was generated.

## 1) Quick start

In [1]:
import autoencodix as acx

config = acx.propose_initial_config("varix")
print(type(config).__name__)
print("epochs:", config.epochs, "| checkpoint_interval:", config.checkpoint_interval)
print("latent_dim:", config.latent_dim, "| beta:", config.beta)

VarixConfig
epochs: 300 | checkpoint_interval: 300
latent_dim: 58 | beta: 0.1497123009757529


With no other arguments, this returns the top-ranked config from the **combined** portfolio
(pooling all BBOmix source tasks across datasets) for the `"downstream"` objective (maximizing
aggregate downstream classification performance, at the full training budget). That combination
is the recommended default: it's the portfolio expected to generalize best to a dataset that
wasn't part of the original benchmark, which is the situation most callers of this function will
actually be in.

## 2) Objectives: `"downstream"` vs. `"reconstruction"`

There are exactly two objective *kinds*:

- `"downstream"` (the default) — maximizes an aggregate downstream classification score. This is
  only ever evaluated at the final epoch (there's no per-epoch downstream signal in the benchmark
  data), so it always recommends the full training budget.
- `"reconstruction"` — minimizes reconstruction loss, which *was* recorded every epoch during the
  sweep. This objective has a genuine fidelity axis: pass `budget_epochs` to get a config tuned
  for training that long, not for 300 epochs.

In [2]:
config_downstream = acx.propose_initial_config("varix", objective="downstream")
config_recon_full = acx.propose_initial_config("varix", objective="reconstruction")
config_recon_short = acx.propose_initial_config(
    "varix", objective="reconstruction", budget_epochs=25
)

print("downstream epochs:                ", config_downstream.epochs)
print("reconstruction (full) epochs:      ", config_recon_full.epochs)
print("reconstruction (budget=25) epochs: ", config_recon_short.epochs,
      "| checkpoint_interval:", config_recon_short.checkpoint_interval)

downstream epochs:                 300
reconstruction (full) epochs:       300
reconstruction (budget=25) epochs:  25 | checkpoint_interval: 25


`budget_epochs` snaps to the nearest of a fixed grid (`10, 25, 50, 100, 150, 200, 300`) and warns
when it does:

In [3]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    config = acx.propose_initial_config("varix", objective="reconstruction", budget_epochs=40)
    print("snapped epochs:", config.epochs)
    print("warning:", caught[0].message)

snapped epochs: 50


Combining `objective="downstream"` with `budget_epochs` raises a `ValueError` — there's nothing
for it to act on.

## 3) Scoping to a benchmark dataset (secondary, optional)

`dataset=None` (the default, used above) is the recommended path for a new dataset. If you
specifically want the portfolio derived from just one of the BBOmix benchmark's own datasets, pass
its name:

In [4]:
config_tcga = acx.propose_initial_config("varix", dataset="tcga")
config_schc = acx.propose_initial_config("varix", dataset="schc")
print(config_tcga.epochs, config_schc.epochs)

300 300


`dataset` is validated dynamically against whatever's actually in the shipped portfolio — not a
fixed set baked into the function signature. An unknown name raises a `ValueError` that tells you
what is available:

In [5]:
try:
    acx.propose_initial_config("varix", dataset="my_new_cohort")
except ValueError as e:
    print(e)

Unknown dataset 'my_new_cohort' for architecture 'varix'. Available datasets: ['schc', 'tcga']. Omit `dataset` (or pass None) to use the combined, cross-dataset portfolio -- the recommended default for a new dataset that wasn't part of the BBOmix benchmark.


## 4) Getting several ranked candidates

In [6]:
candidates = acx.propose_initial_config("varix", top_k=3)
for i, c in enumerate(candidates, 1):
    print(i, "latent_dim:", c.latent_dim, "| n_layers:", c.n_layers, "| lr:", c.learning_rate)

1 latent_dim: 58 | n_layers: 3 | lr: 0.0033023196273975707
2 latent_dim: 15 | n_layers: 4 | lr: 8.629522298260548e-05
3 latent_dim: 57 | n_layers: 3 | lr: 0.0003654758769476195


## 5) Overriding fields

Any keyword you pass through `**overrides` is applied on top of the proposed hyperparameters
before the Config object is constructed — so you can take the proposal as a base and tweak just
what you care about. Pydantic still validates the result, so an out-of-range override raises the
normal `ValidationError`.

In [7]:
from pydantic import ValidationError

config = acx.propose_initial_config("varix", latent_dim=8, learning_rate=1e-3)
print(config.latent_dim, config.learning_rate)

try:
    acx.propose_initial_config("varix", latent_dim=-1)
except ValidationError as e:
    print("rejected:", e.errors()[0]["msg"])

8 0.001
rejected: Input should be greater than or equal to 1


## 6) Architectures without a BBOmix portfolio

Only `vanillix`, `varix`, `ontix`, and `disentanglix` have a BBOmix-derived portfolio. Other
architectures with a Config class (e.g. `stackix`, `xmodalix`, `maskix`) raise a `ValueError` by
default; pass `allow_fallback_to_defaults=True` to instead get that class's plain schema defaults
with a warning.

In [8]:
try:
    acx.propose_initial_config("stackix")
except ValueError as e:
    print(e)

config = acx.propose_initial_config("stackix", allow_fallback_to_defaults=True)
print(type(config).__name__, "epochs:", config.epochs)

Unsupported architecture 'stackix'. propose_initial_config has a BBOmix-derived portfolio for: ['disentanglix', 'ontix', 'vanillix', 'varix']. Pass allow_fallback_to_defaults=True to fall back to plain schema defaults for any other architecture with a Config class.
StackixConfig epochs: 3


/var/folders/fz/3nckfp3n60s2f4p0_rb0vbph0000gn/T/ipykernel_4334/2371638015.py:6: UserWarning: No BBOmix-derived portfolio for architecture 'stackix'; falling back to StackixConfig schema defaults. Supported architectures with a portfolio: ['disentanglix', 'ontix', 'vanillix', 'varix'].
  config = acx.propose_initial_config("stackix", allow_fallback_to_defaults=True)


## What's next

This function only ever proposes an *initial* config, so it has no way to learn from your own
training runs. To optimize your model further you can use actual HPO methods. These steps are explained for Syne Tune in `HyperparameterOptimizationTutorial.ipynb` and Optuna in `HyperparameterOptimizationOptunaTutorial.ipynb`.